In [ ]:
import sympy as sp

# We will first write the equation comming from a_0, a_1 and b_0

In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.special import jv  # Bessel J_n

def solve_system_fourier(beta_a_p, beta_b_p, gamma, kappa_b_p, eps_d_p,
                         m_max=10,
                         a0_init=0.626, b0_init=0.476,
                         a1re_init=-0.0903017258275538,
                         theta_a0_init=0.0, theta_b0_init=-0.13098445758448743):
    """
    Unknowns: |a0|, |b0|, Re(a1), theta_a0, theta_b0  (5 real unknowns)

    Equations (truncate sum over m in [-m_max, m_max]):

      (E1) 0 = sum_m (-1)^{m+1} J_m(2|b0|) J_{2m+1}(2|a0|)
               * exp(i m (2θa - θb)) * sin(2γ Re(a1) + mπ/2)

      (E2) 1 + Re(a1)/βa' = sum_m (-1)^m J_m(2|b0|) J_{2m}(2|a0|)
               * exp(i m (2θa - θb)) * cos(2γ Re(a1) + mπ/2)

      (E3) i κb' |b0| e^{iθb} - εd'/2
            = βb' e^{iθb} * sum_m (-1)^m J_m(2|b0|) J_{2m+2}(2|a0|)
               * exp(i m (2θa - θb)) * cos(2γ Re(a1) + mπ/2)

    We solve 5 real equations:
      Re(E1)=0, Im(E1)=0, Re(E2- (1+a1/βa))=0, Im(E2)=0, Re(E3)=0, Im(E3)=0
    but note E2 should be real by symmetry; we enforce its real part only and also
    enforce Im(E2)=0 to stabilize (total 6). To keep 5 unknowns, we drop Im(E2)
    and solve Re(E1)=0, Im(E1)=0, Re(E2 - (1+a1/βa))=0, Re(E3)=0, Im(E3)=0.
    """

    m_vals = np.arange(-m_max, m_max + 1)

    def sums(a0, b0, a1re, theta_a0, theta_b0):
        phase = np.exp(1j * m_vals * (2.0*theta_a0 - theta_b0))
        arg = 2.0*gamma*a1re + m_vals * (np.pi/2)

        Jm_b = jv(m_vals, 2.0*b0)

        S1 = np.sum(((-1.0)**(m_vals+1)) * Jm_b * jv(2*m_vals+1, 2.0*a0) * phase * np.sin(arg))
        S2 = np.sum(((-1.0)**(m_vals))   * Jm_b * jv(2*m_vals,   2.0*a0) * phase * np.cos(arg))
        S3 = np.sum(((-1.0)**(m_vals))   * Jm_b * jv(2*m_vals+2, 2.0*a0) * phase * np.cos(arg))
        return S1, S2, S3

    def F(v):
        a0, b0, a1re, theta_a0, theta_b0 = v

        S1, S2, S3 = sums(a0, b0, a1re, theta_a0, theta_b0)

        # Eq (1): S1 = 0 (complex) -> Re=0, Im=0
        e1_re = np.real(S1)
        e1_im = np.imag(S1)

        # Eq (2): 1 + a1/βa = S2  -> enforce real part
        e2_re = np.real(S2) - (1.0 + a1re/beta_a_p)

        # Eq (3): i κb' b0 e^{iθb} - εd'/2 = βb' e^{iθb} S3 (complex)
        lhs3 = 1j * kappa_b_p * b0 * np.exp(1j*theta_b0) - eps_d_p/2.0
        rhs3 = beta_b_p * np.exp(1j*theta_b0) * S3
        e3 = lhs3 - rhs3
        e3_re = np.real(e3)
        e3_im = np.imag(e3)

        return np.array([e1_re, e1_im, e2_re, e3_re, e3_im], dtype=float)

    x0 = np.array([a0_init, b0_init, a1re_init, theta_a0_init, theta_b0_init], dtype=float)

    sol = root(F, x0, method="hybr", tol=1e-12)

    if not sol.success:
        raise RuntimeError(f"Root solver failed: {sol.message}")

    a0, b0, a1re, theta_a0, theta_b0 = sol.x

    return {
        "abs_a0": a0,
        "abs_b0": b0,
        "Re_a1": a1re,
        "theta_a0": theta_a0,
        "theta_b0": theta_b0,
        "residual": np.linalg.norm(F(sol.x), ord=2),
        "nfev": sol.nfev,
        "m_max": m_max,
    }

# Example:
# out = solve_system_fourier(beta_a_p=..., beta_b_p=..., gamma=..., kappa_b_p=..., eps_d_p=..., m_max=10)
# print(out)


In [ ]:
# Example usage
beta_a_p = 0.1831084342479521
beta_b_p = 0.3148859751926766
gamma = 1+beta_b_p/beta_a_p
epsilon_d_p = 0.009779030042836782
kappa_b_p = 0.0020537287677222364

out = solve_system_fourier(beta_a_p=beta_a_p, beta_b_p=beta_b_p, gamma=gamma, kappa_b_p=kappa_b_p, eps_d_p=epsilon_d_p, m_max=10)
print(out)


In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.special import jv  # Bessel J_n

def solve_system_mp(beta_a_p, beta_b_p, gamma, kappa_b_p, eps_d_p,
                    m_max=8, p_max=8,
                    a0_init=0.47, b0_init=0.34, a1_init=-0.08, bm1_init=-0.07,
                    cos_branch=+1):
    """
    Solve for (|a0|, |b0|, a1(real), |b_-1|) using truncated double sums over m,p.

    cos_branch: +1 uses cos(theta_b0)=+sqrt(1-sin^2), -1 uses negative branch.
    """

    m_vals = np.arange(-m_max, m_max + 1)
    p_vals = np.arange(-p_max, p_max + 1)

    # For speed: prebuild 2D grids of m,p
    M, P = np.meshgrid(m_vals, p_vals, indexing="ij")  # shape (Nm, Np)

    # (-1)^m  (note: (-1)^(3m)=(-1)^m)
    sgn_m = (-1.0) ** M

    def cos_theta_b0(b0):
        s = -2.0 * kappa_b_p * b0 / eps_d_p
        # If |s|>1 there is no real theta; root will struggle; return nan to signal failure
        if np.any(np.abs(s) > 1.0):
            return np.nan, np.nan
        c = cos_branch * np.sqrt(1.0 - s**2)
        return s, c

    def sums(a0, b0, a1, bm1):
        # Compute all needed Bessel factors on the (m,p) grid
        Jm_b0 = jv(M, 2.0 * b0)
        Jp_bm1 = jv(P, 2.0 * bm1)

        # Common phase arguments
        arg0 = 2.0 * gamma * a1

        # Indices for J_{2m+3p+q}(2|a0|)
        k1 = 2*M + 3*P + 1
        k2 = 2*M + 3*P + 0
        k3 = 2*M + 3*P + 2
        k4 = 2*M + 3*P + 3

        Jk1_a0 = jv(k1, 2.0 * a0)
        Jk2_a0 = jv(k2, 2.0 * a0)
        Jk3_a0 = jv(k3, 2.0 * a0)
        Jk4_a0 = jv(k4, 2.0 * a0)

        # Cosine arguments: (m+2p+shift)*pi/2
        c1 = np.cos(arg0 + (M + 2*P + 1) * (np.pi/2))
        c2 = np.cos(arg0 + (M + 2*P + 0) * (np.pi/2))
        c3 = np.cos(arg0 + (M + 2*P + 2) * (np.pi/2))
        c4 = np.cos(arg0 + (M + 2*P + 3) * (np.pi/2))

        # Double sums
        S1 = np.sum(sgn_m * Jm_b0 * Jk1_a0 * Jp_bm1 * c1)
        S2 = np.sum(sgn_m * Jm_b0 * Jk2_a0 * Jp_bm1 * c2)
        S3 = np.sum(sgn_m * Jm_b0 * Jk3_a0 * Jp_bm1 * c3)  # since (-1)^(3m)=(-1)^m
        S4 = np.sum(sgn_m * Jm_b0 * Jk4_a0 * Jp_bm1 * c4)

        return S1, S2, S3, S4

    def F(v):
        a0, b0, a1, bm1 = v

        sin_th, cos_th = cos_theta_b0(b0)
        if not np.isfinite(cos_th):
            return np.array([1e6, 1e6, 1e6, 1e6], dtype=float)

        S1, S2, S3, S4 = sums(a0, b0, a1, bm1)

        # Eq(1)
        eq1 = S1

        # Eq(2)
        eq2 = S2 - (1.0 + a1 / beta_a_p)

        # Eq(3):  -kappa b0 sin(theta) - eps/2 = beta_b cos(theta) * S3
        eq3 = (-kappa_b_p * b0 * sin_th - eps_d_p/2.0) - (beta_b_p * cos_th * S3)

        # Eq(4):  bm1 = 2 beta_b * S4
        eq4 = bm1 - 2.0 * beta_b_p * S4

        return np.array([eq1, eq2, eq3, eq4], dtype=float)

    x0 = np.array([a0_init, b0_init, a1_init, bm1_init], dtype=float)
    sol = root(F, x0, method="hybr", tol=1e-12)

    if not sol.success:
        raise RuntimeError(f"Root solver failed: {sol.message}")

    a0, b0, a1, bm1 = sol.x
    sin_th, cos_th = cos_theta_b0(b0)
    theta_b0 = np.arctan2(sin_th, cos_th)  # consistent with chosen branch

    return {
        "abs_a0": float(a0),
        "abs_b0": float(b0),
        "a1": float(a1),
        "abs_bm1": float(bm1),
        "theta_b0": float(theta_b0),
        "sin_theta_b0": float(sin_th),
        "cos_theta_b0": float(cos_th),
        "residual_L2": float(np.linalg.norm(F(sol.x), 2)),
        "nfev": int(sol.nfev),
        "m_max": int(m_max),
        "p_max": int(p_max),
        "cos_branch": int(cos_branch),
    }

# Example:
# out = solve_system_mp(beta_a_p=..., beta_b_p=..., gamma=..., kappa_b_p=..., eps_d_p=...,
#                       m_max=8, p_max=8, a0_init=0.626, b0_init=0.476, a1_init=-0.0903, bm1_init=0.1,
#                       cos_branch=+1)
# print(out)


In [ ]:
out = solve_system_mp(beta_a_p=beta_a_p, beta_b_p=beta_b_p, gamma=gamma, kappa_b_p=kappa_b_p, eps_d_p=epsilon_d_p,
                      m_max=20, p_max=20, a0_init=0.626, b0_init=0.476, a1_init=-0.0903, bm1_init=0.1,
                      cos_branch=+1)
print(out)